# [실습] LangChain으로 첫 체인 만들기

## 라이브러리 설치  

랭체인 관련 라이브러리를 설치합니다.  


In [78]:
# LangChain + LLM.......$@%^#*
!pip install langchain-huggingface langchain langchain-openai langchain-community langchain-ollama langchain-core langchain-google-genai #langchain-google-generativeai
!pip install dotenv langchain-chroma

## 환경 변수 설정하기


In [79]:
import os
from dotenv import load_dotenv
# OPENAI_API_KEY, GOOGLE_API_KEY
load_dotenv(".env", override=True)

True

모델 Provide의 클래스를 불러온 뒤, LLM을 연결하면 됩니다!

## Ollama를 통해 LLM 불러오기


https://ollama.com/


올라마는 가장 간단하게 오픈 모델을 구동할 수 있는 방식입니다.   
추론 최적화/병렬 처리 등의 기능은 뛰어나지 않지만  
CPU에서도 구동이 가능하며, UI도 지원합니다.

### Ollama 설치하고 서빙하기    
터미널을 이용해 아래의 커맨드를 입력합니다.

- 설치

```curl -fsSL https://ollama.com/install.sh | sh```

- Context Window 늘리기 (기본 Context 4096)

```export OLLAMA_CONTEXT_LENGTH=16384```

- 서빙 (& 으로 백그라운드 실행)

``` ollama serve &```



In [ ]:
#!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
ERROR: This version requires zstd for extraction. Please install zstd and try again:
  - Debian/Ubuntu: sudo apt-get install zstd
  - RHEL/CentOS/Fedora: sudo dnf install zstd
  - Arch: sudo pacman -S zstd


In [80]:
!sudo apt-get install zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.


In [81]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [82]:
!ollama --version
!export OLLAMA_CONTEXT_LENGTH=16384
#!ollama serve &

올라마는 기본적으로 11434 포트에서 실행됩니다.   
Pull 을 통해 허깅페이스 모델의 주소나 Ollama 주소를 입력하면 모델을 불러와 저장합니다.

불러올 수 있는 모델의 목록은 https://ollama.com/search 에서 확인할 수 있습니다.   

기본적으로 양자화를 거친 상태로 불러오며, 16비트 Full Precision을 비롯한 다양한 버전도 확인할 수 있습니다.

터미널에서 아래 커맨드를 입력해 GPT-OSS 20B 모델을 불러옵니다.    
```
ollama pull gpt-oss:20b
```

In [ ]:
#!ollama pull gpt-oss:20b

Error: could not connect to ollama server, run 'ollama serve' to start it


In [83]:
import subprocess
import time

ollama_process = subprocess.Popen(["ollama", "serve"],
                                  stdout=subprocess.PIPE,
                                  stderr=subprocess.PIPE
                                  )


time.sleep(10)
print("Ollama server started")

Ollama server started


In [84]:
!ollama pull gpt-oss:20b

불러온 모델은 파일 시스템에 저장되며,   
Ollama 모듈을 호출할 때 해당 모델이 GPU에 로드되는 구조입니다.

In [85]:
#import sys
#!{sys.executable} -m pip install langchain-ollama
from langchain_ollama import ChatOllama
llm = ChatOllama(model='gpt-oss:20b',
                 temperature=0.1, reasoning=True)

test = '안녕!!!!!'
llm.invoke(test)

AIMessage(content='안녕! 😊 오늘은 어떤 도움을 드릴까? 궁금한 점이나 필요한 정보가 있으면 언제든 물어봐~', additional_kwargs={'reasoning_content': 'The user says "안녕!!!!!" which is Korean greeting meaning "Hello!" with excitement. The assistant should respond in Korean presumably. It\'s a friendly greeting. We can reply with something like "안녕하세요! 어떻게 도와드릴까요?" or similar. Let\'s keep it friendly and ask how to help.'}, response_metadata={'model': 'gpt-oss:20b', 'created_at': '2026-06-29T02:14:21.948380454Z', 'done': True, 'done_reason': 'stop', 'total_duration': 9174891822, 'load_duration': 7821285925, 'prompt_eval_count': 70, 'prompt_eval_duration': 377564000, 'eval_count': 101, 'eval_duration': 960041000, 'logprobs': None, 'model_name': 'gpt-oss:20b', 'model_provider': 'ollama'}, id='lc_run--019f1127-dc24-7a11-90fb-9fdb88ef60dd-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 70, 'output_tokens': 101, 'total_tokens': 171})

In [ ]:
test = 'ㅠㅠ'
for s in llm.stream(test):
    print(s.content, end='')

## Embedding


RAG의 구조는 검색을 수행하기 위한 임베딩 모델이 필요합니다.   
임베딩 모델은 텍스트를 벡터로 변환하며,    
이후 결과를 벡터 DB에 저장해 검색할 수 있습니다.

OpenAI의 `text-embedding-3-large` 는 빠른 속도로 연산이 가능하나, 비용이 발생하며 온라인 모델입니다.   
이에 따라, 폐쇄망/온프레미스 환경에서는 공개 임베딩 모델을 사용하여 구현해야 합니다.

In [86]:
from langchain_openai import OpenAIEmbeddings
openai_embeddings = OpenAIEmbeddings(model='text-embedding-3-large')

허깅페이스에 게시된 공개 모델을 불러옵니다.   
오픈 임베딩 모델에서 중요한 파라미터는 다음과 같습니다.

- 파라미터 수 : 큰 임베딩 모델의 크기는 LLM에 육박합니다. GPU를 고려하여 선택합니다.
- Max Tokens: 임베딩 모델의 최대 토큰보다 큰 데이터를 입력하면, 앞부분만을 이용해 계산하게 되므로 적절한 검색이 되지 않을 수 있습니다.
- 임베딩 차원: 큰 차원의 벡터를 생성하는 임베딩 모델은 검색 속도가 감소합니다.

현재 한국어 데이터를 임베딩하기 위해 자주 사용하는 모델은 아래와 같습니다.

- BAAI/BGE-M3 (2GB, 8194 토큰 제한)   
BGE-M3 시리즈는 BAAI의 임베딩 모델로, 현재 가장 인기가 많은 모델입니다.

- nlpai-lab/KURE-v1 (2GB, 8194 토큰 제한)    
KURE 임베딩은 고려대학교 NLP 연구실에서 만든 모델로, BGE-M3를 한국어 텍스트로 파인 튜닝한 모델입니다.

- Qwen/Qwen-3-Embedding (0.6B, 4B, 8B, 32768 토큰 제한)    
알리바바 클라우드의 Qwen 모델을 개량하여 만든 모델입니다.    
가장 최신 모델로, BGE-M3와의 성능 비교가 치열합니다.

In [87]:
from sentence_transformers import SentenceTransformer
import torch

# HuggingFace 임베딩 주소 지정하기
# intfloat/multilingual-e5-small , baai/bge-m3, 등의 주소를 입력하여 지정
# GPU에 여유가 있다면 Qwen3 Embedding의 큰 사이즈 (4B, 8B)?

#model_name = 'Qwen/Qwen3-Embedding-0.6B'
model_name = 'Qwen/Qwen3-Embedding-4B'
#실제 주소: https://huggingface.co/BAAI/bge-m3

# CPU 설정으로 모델 불러오기
emb_model = SentenceTransformer(model_name,
            device='cuda', #'cpu',
            model_kwargs={'torch_dtype':torch.bfloat16})

# 로컬 폴더에 모델 저장하기
emb_model.save('./embedding')

# 모델 메모리에서 삭제
del emb_model
import gc
gc.collect()

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

9617

파일 시스템에 저장한 오픈 모델은 HuggingFaceEmbeddings로 불러옵니다.

In [88]:
from langchain_huggingface import HuggingFaceEmbeddings
import torch

# 허깅페이스 포맷의 임베딩 모델 불러오기
open_embeddings = HuggingFaceEmbeddings(model_name= './embedding',
                                   model_kwargs={'device':'cuda', 'model_kwargs':{'torch_dtype':torch.bfloat16}})
                                                    # gpu 사용하기                               bfloat16

# GPU 로드된 것 확인
print('임베딩 모델 로드 완료!')

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

임베딩 모델 로드 완료!


RAG를 하기 전, 비교를 위해 LLM에게 질문해 보겠습니다.

In [89]:
# Test
thinking = False
for s in llm.stream("도메인 특화 언어 모델이란 무엇입니까? 어떤 예시가 있나요?"):
    if s.additional_kwargs.get('reasoning_content'):
        if not thinking:
            print('<THINK> ', end='')
            thinking=True
        print(s.additional_kwargs['reasoning_content'], end='')
    else:
        if thinking:
            print('</THINK>\n\n', end='')
            thinking=False
        print(s.content,end='')

<THINK> The user asks in Korean: "What is a domain-specific language model? What are some examples?" They want explanation of what a domain specialized language model is, and examples. Provide definition, characteristics, benefits, typical use cases, and examples like BioBERT (biomedical), ClinicalBERT, FinBERT for finance, LegalBERT, CodeBERT for programming, etc. Also mention GPT-4 fine-tuned on specific domains. Provide explanation in Korean. Should be thorough but concise.</THINK>

## 도메인 특화 언어 모델(Domain‑Specific Language Model)이란?

### 1. 정의
**도메인 특화 언어 모델**은 일반적인 대규모 언어 모델(예: GPT, BERT 등)을 특정 분야(도메인)의 텍스트에 맞춰 추가 학습(fine‑tuning)하거나 처음부터 그 도메인의 데이터만으로 훈련시킨 모델입니다.  
- **특징**  
  - 해당 도메인에서 자주 등장하는 전문 용어, 문체, 구조를 잘 파악합니다.  
  - 일반 모델보다 같은 작업(예: 질의응답, 요약, 분류)에서 더 높은 정확도와 신뢰성을 제공합니다.  
  - 데이터가 충분히 확보된 분야에서 특히 효과적입니다.

### 2. 왜 필요한가?
| 문제 | 일반 모델이 겪는 어려움 | 도메인 특화 모델이 해결하는 방식 |
|------|------------------------|-----------------------------------|
| **전문 용어** | 드물게 등장하거나 의미가 모호 | 대량의 전문 

## 데이터 준비하기    
네이버 API를 통해, 검색어에 대한 뉴스 기사 링크를 가져오겠습니다.    


In [90]:
import requests
import os

def get_naver_news_links(query, num_links=100):
    """
    query와 num_links를 입력받아 네이버 검색 수행, 네이버 뉴스 URL의 기사만 수집
    """

    url = f"https://openapi.naver.com/v1/search/news.json?query={query}&display={num_links}&sort=sim"
    # 최대 100개의 결과를 표시

    NAVER_CLIENT_ID = 'g4yR7k_1CLoMOogWn3Q8'
    NAVER_CLIENT_SECRET = 'e9hdJO8V_2'

    client_id = NAVER_CLIENT_ID # os.getenv('NAVER_CLIENT_ID')
    client_secret = NAVER_CLIENT_SECRET # os.getenv('NAVER_CLIENT_SECRET')

    headers = {
        'X-Naver-Client-Id': client_id,
        'X-Naver-Client-Secret': client_secret
    }

    response = requests.get(url, headers=headers)
    result = response.json()

    # 특정 링크 형식만 필터링
    filtered_links = []
    for item in result['items']:
        link = item['link']
        if "n.news.naver.com/mnews/article/" in link:
            # 네이버 뉴스 스타일만 모으기
            filtered_links.append(link)

    # 결과 출력
    print(query, ':', len(filtered_links), 'Example:', filtered_links[0])
    # for link in filtered_links:
    #     print(link)

    return filtered_links

filtered_links = []
for topic in ['도메인 특화 언어모델', 'OpenAI', 'GPT', '구글', '가전제품', '넷플릭스']:
    filtered_links += get_naver_news_links(topic, 100)
print('Total Articles:', len(filtered_links))
print('Total Articles(Without Duplicate):',len(list(set(filtered_links))))
filtered_links = list(set(filtered_links))

도메인 특화 언어모델 : 47 Example: https://n.news.naver.com/mnews/article/138/0002232009?sid=105
OpenAI : 25 Example: https://n.news.naver.com/mnews/article/081/0003656388?sid=101
GPT : 78 Example: https://n.news.naver.com/mnews/article/009/0005699686?sid=105
구글 : 70 Example: https://n.news.naver.com/mnews/article/001/0016164278?sid=104
가전제품 : 56 Example: https://n.news.naver.com/mnews/article/022/0004138884?sid=101
넷플릭스 : 32 Example: https://n.news.naver.com/mnews/article/018/0006317488?sid=103
Total Articles: 308
Total Articles(Without Duplicate): 303


## LangChain Document Loaders

LangChain의 `document_loaders`는 다양한 형식의 파일을 불러올 수 있습니다.   
[https://python.langchain.com/docs/integrations/document_loaders/ ]    

Web URL로부터 페이지를 로드하는 기본 파서인 `WebBaseLoader`를 사용합니다.   

In [91]:
# Jupyter 분산 처리를 위한 설정
import nest_asyncio

nest_asyncio.apply()

In [92]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

async def get_news_documents(links):
    loader = WebBaseLoader(
        web_paths=links,
        bs_kwargs={'parse_only':bs4.SoupStrainer(class_=("newsct", "newsct-body"))},
                                # newsct, newsct-body만 추출 : 네이버 뉴스 포맷 HTML 요소

        requests_per_second = 10, # 1초에 10개 요청 보내기
        show_progress = True # 진행 상황 출력
    )
    # docs = loader.load() # 기본 코드
    # return docs

    docs = []

    async for doc in loader.alazy_load():
        # 순차적 로드 대신 비동기 처리
        docs.append(doc)
    return docs

docs = await get_news_documents(filtered_links)

Fetching pages: 100%|##########| 303/303 [02:08<00:00,  2.36it/s]


In [ ]:
from rich import print as rprint
rprint(docs[12])

Document(
    metadata={'source': 'https://n.news.naver.com/mnews/article/374/0000518676?sid=104'},
    page_content='\n\n\n\n\n\n\nSBS Biz\n\n\n구독\n\nSBS Biz 언론사 구독되었습니다. 메인 뉴스판에서  주요뉴스를  볼
수 있습니다.\n보러가기\n닫기\n\n\nSBS Biz 언론사 구독 해지되었습니다.\n닫기\n\n\n\n\n[글로벌 비즈 브리핑] 오픈AI, 
새 AI 모델 공개…美 정부 요청에 일부기관 제공 外\n\n\n\n\n\n\n\n\n입력\n2026.06.29. 오전 3:29\n\n\n수정\n2026.06.29.
오전 5:48\n\n\n\n기사원문\n 
\n\n\n\n\n\n\n\n\n\n\n추천\n반응\n\n\n\n\n쏠쏠정보\n0\n\n\n\n\n흥미진진\n0\n\n\n\n\n공감백배\n0\n\n\n\n\n분석탁월\n
0\n\n\n\n\n후속강추\n0\n\n\n \n\n\n\n\n댓글\n반응\n\n\n\n\n\n\n\n\n텍스트 음성 변환 서비스 
사용하기\n\n\n\n성별\n남성\n여성\n\n\n말하기 속도\n느림\n보통\n빠름\n\n이동 통신망을 이용하여 음성을 재생하면 
별도의 데이터 통화료가 부과될 수 있습니다.\n본문듣기 시작\n\n닫기\n\n\n \n\n글자 크기 
변경하기\n\n글자크기\n\n\n가1단계\n작게\n\n\n가2단계\n보통\n\n\n가3단계\n크게\n\n\n가4단계\n아주크게\n\n\n가5단계\n
최대크게\n\n\n닫기\n\n\n\n\nSNS 보내기\n\n\n\n인쇄하기\n\n\n\n\n\n\n\n\n\n\n\n\n[오픈AI 로고 일러스트 
(로이터=연합뉴스)][글로벌 비즈 브리핑] 한 눈에 보는 해외 경제 이슈▲메모리 대란에...애플 "中 칩 구해달라" 美 정부에 
로비▲오픈AI, 새 AI 모델 공개...美 정부 요청에 일부기관 제공▲"AI 붐, 장기 투자 실패로 끝날 위험:...섬뜩한 
경고▲한사람당 100억원 벌었다...日 키옥시아 직원들 \'잭팟\'▲스페이스X, 모바일 통신사업 진출 시동...차터와 
논의▲"비트코인, 소리없이 사라질 것"...닷컴버블 맞힌 그랜섬의 경고메모리 대란에...애플 "中 칩 구해달라" 美 정부에 
로비세계적인 메모리 반도체 품귀 탓에 생산 비용이 크게 늘어난 애플이 미국 정부의 블랙리스트에 오른 중국산 메모리 칩 
구매를 적극 검토하는 것으로 전해졌습니다.현지시간 26일 파이낸셜타임스(FT)는 애플이 미국 상무부를 비롯한 도널드 
트럼프 행정부 관계자들을 상대로 중국산 메모리 칩 구매를 승인해달라는 로비를 벌이고 있다고 보도했습니다.애플이 
관심을 보이는 중국의 반도체 업체는 창신메모리테크놀로지(CXMT)인 것으로 알려졌습니다.D램 제조사인 CXMT는 중국 
인민해방군과 연계됐다는 이유로 미 국방부가 ‘중국 군사기업(1260H)’이라는 일종의 블랙리스트에 올려놓은 업체입니다.이 
명단에 포함될 경우 직접적인 법적 제재를 받습니다. 하지만 거래 기업은 평판 리스크를 감수해야 합니다.애플이 워싱턴 
정가를 상대로 로비에 나선 것은 메모리 가격 급등으로 공급망 다변화를 추진해야 하는 상황이기 때문입니다.그러나 애플의
로비가 성공할지 여부는 미지수입니다. 특히 미국 의회의 반발이 변수로 꼽힙니다.하원 중국특별위원회의 존 
물레나(공화·미시간) 위원장은 “애플이 중국 군사 기업과 협력하는 것은 심각한 실수가 될 것”이라며 “중국 공산당이 핵심 
공급망을 장악하도록 돕는 결과를 초래할 수 있다”고 비판했습니다.마코 루비오 국무장관도 상원의원이던 지난 2022년 
애플이 중국의 YMTC 메모리 칩 채택을 검토했을 당시 “불장난을 하는 것”이라며 강하게 반대한 바 있습니다.낸드플래시 
메모리 전문 업체인 YMTC는 중국 군사기업 블랙리스트에 올라와 있는 업체입니다.오픈AI, 새 AI 모델 공개...美 정부 
요청에 일부기관 제공오픈AI가 차세대 인공지능(AI) 모델 \'GPT-5.6\' 제품을 출시하면서, 미국 정부의 요청으로 일부 
기관에만 제한적으로 우선 제공하기로 했습니다.오픈AI는 새 AI 모델 \'GPT-5.6\'을 \'솔\', \'테라\', \'루나\' 등 세 
가지 세부모델로 나눠 개발했다고 현지시간 26일 밝혔습니다.이 가운데 상위 모델인 \'솔\' 모델은 코딩, 생물학, 사이버 
보안 등 부문에서 역대 최고 성능과 자율적 에이전트 능력을 갖췄다고 오픈AI는 설명했습니다.오픈AI는 이 모델의 추론 
능력을 강화하기 위해 모델에 역대 가장 긴 시간을 부여하는 \'최대 추론 노력\' 옵션과 하위 에이전트를 활용해 복잡한 
작업을 좀 더 빠르게 수행하는 \'울트라 모드\'도 새로 도입했습니다.오픈AI는 터미널 코딩 능력을 재는 \'터미널-벤치 
2.1\' 성능지표(벤치마크)에서 GPT-5.6 솔 울트라가 91.9%, GPT-5.6 솔이 88.8%를 기록해 라이벌 앤트로픽의 클로드 
미토스5의 88%를 능가했다고도 강조했습니다.특히 오픈AI는 솔 모델이 현재까지 가장 견고한 안전 체계를 갖춰 고위험 
활동이나 해킹에 악용될 우려가 있는 민감한 사이버 요청, 오용 등에 대한 보호 조치를 강화했다고도 
강조했습니다.오픈AI는 그간 새 모델 발표 때마다 제공해 왔던 세부적인 성능지표는 공개하지 않았습니다.중위 세부 모델인
\'테라\'는 전작인 GPT-5.5와 유사한 성능을 보이면서도 가격은 절반으로 낮췄고, 하위 모델인 \'루나\'는 최저 비용으로 
준수한 성능을 제공한다고 밝혔습니다.오픈AI는 이들 모델을 우선 정부와 공유된 \'신뢰할 수 있는 파트너\'를 대상으로 선
제공한다면서 실제 일반 배포 시기는 몇 주 뒤가 될 것이라고 예고했습니다.다만 오픈AI는 "이러한 형태의 정부 승인 
절차가 장기적인 표준이 되어서는 안 된다고 생각한다"며 "해당 도구가 필요한 사용자·개발자·기업·사이버 보안 
전문가·글로벌 파트너들이 최고의 도구를 활용하지 못하게 막기 때문"이라고 덧붙였습니다.도널드 트럼프 대통령은 이달 초
AI 기업이 새 모델을 공개하기 최대 30일 전에 정부에 제출할 수 있도록 하는 내용의 행정명령에 서명했고, 미 상무부는 
앤트로픽의 첨단 모델 \'미토스5\'와 \'페이블5\'가 국가 안보 위험을 초래한다며 수출 통제 지침을 내놨습니다.한편 
오픈AI는 당초 올해로 예상됐던 상장 추진 일정을 늦추고 있는 것으로 전해졌습니다.이달 초 증권거래위원회(SEC)에 비공개
기업공개(IPO) 신청서를 제출한 오픈AI는 아직 잠재적인 공모 가격과 수요 등을 논의하기 위한 사전 회의도 열지 않았고, 
상장 일정에 대한 공식 계획도 제시하지 않았다고 미 경제방송 CNBC가 소식통을 인용해 이날 보도했습니다.앞서 
뉴욕타임스(NYT)는 오픈AI가 연내 추진하려 했던 상장 시기를 내년으로 미루는 쪽으로 기울고 있다고 전날 보도한 바 
있습니다."AI 붐, 장기 투자 실패로 끝날 위험:...섬뜩한 경고빅테크의 인공지능(AI) 투자 광풍이 장기적인 투자 붕괴로 
끝날 위험이 있다고 국제결제은행(BIS)이 경고했습니다.28일(현지시간) 파이낸셜타임스(FT)에 따르면 BIS는 이날 ‘연례 
경제 보고서’를 통해 기술 부문의 수익률이 기대에 못 미칠 가능성이 커질 경우 투자자들이 AI 기업에 대한 자금 공급을 
급격히 줄일 수 있다고 내다봤습니다. 5대 대형 하이퍼스케일러(초대형 클라우드 사업자)들은 2025년부터 2026년 말까지 
1조달러 이상을 투자할 것으로 예상됩니다.BIS는 현재의 AI 호황이 안고 있는 위험을 설명하며 “수익률에 대한 실망은 자금
조달의 갑작스러운 후퇴를 촉발할 수 있고, 자본지출 붐을 장기적인 투자 침체로 바꿀 수 있다”며 “이는 금융 여건에도 
연쇄적인 영향을 미칠 수 있다”고 분석했습니다.이는 AI 투자를 위한 주식·채권 발행 규모가 커지고, 이로 인해 글로벌 
시장의 변동성이 확대되는 데 대한 우려가 커지는 가운데 나왔습니다. 기술기업들은 AI 프로젝트 자금을 마련하기 위해 
신용시장으로 몰려들어 수천억달러를 조달했습니다.사상 최고 수준의 주가도 이들을 미국 주식시장으로 끌어들이고 
있습니다. 이달 초 스페이스X가 860억달러 규모

불필요한 내용을 전처리합니다.

In [93]:
import re

def preprocess(docs):
    noise_texts = [
        '''구독중 구독자 0 응원수 0 더보기''',
        '''쏠쏠정보 0 흥미진진 0 공감백배 0 분석탁월 0 후속강추 0''',
        '''댓글 본문 요약봇 본문 요약봇''',
        '''도움말 자동 추출 기술로 요약된 내용입니다. 요약 기술의 특성상 본문의 주요 내용이 제외될 수 있어, 전체 맥락을 이해하기 위해서는 기사 본문 전체보기를 권장합니다. 닫기''',
        '''텍스트 음성 변환 서비스 사용하기 성별 남성 여성 말하기 속도 느림 보통 빠름''',
        '''이동 통신망을 이용하여 음성을 재생하면 별도의 데이터 통화료가 부과될 수 있습니다. 본문듣기 시작''',
        '''닫기 글자 크기 변경하기 가1단계 작게 가2단계 보통 가3단계 크게 가4단계 아주크게 가5단계 최대크게 SNS 보내기 인쇄하기''',
        'PICK 안내 언론사가 주요기사로선정한 기사입니다. 언론사별 바로가기 닫기',
        '응원 닫기',
        '구독 구독중 구독자 0 응원수 0 ',

    ]

    def clean_text(doc):
        text = doc.page_content
        # 탭과 개행문자를 공백으로 변환
        text = text.replace('\t', ' ').replace('\n', ' ')

        # 연속된 공백을 하나로 치환
        text = re.sub(r'\s+', ' ', text).strip()

        # 여러 구분자를 한번에 처리
        split_markers = [
            '구독 해지되었습니다.',
            '구독 메인에서 바로 보는 언론사 편집 뉴스 지금 바로 구독해보세요!'
        ]


        for marker in split_markers:
            parts = text.split(marker)
            if len(parts) > 1:
                if marker == '구독 해지되었습니다.':
                    text = parts[1]  # 뒷부분 사용
                else:
                    text = parts[0]  # 앞부분 사용


        # 노이즈 텍스트 제거
        for noise in noise_texts:
            text = text.replace(noise, '')

        # 연속된 공백을 하나로 치환
        text = re.sub(r'\s+', ' ', text).strip()

        doc.page_content = text

        return doc


    preprocessed_docs = []
    for doc in docs:

        # 텍스트 정제
        doc= clean_text(doc)
        preprocessed_docs.append(doc)

    return preprocessed_docs

preprocessed_docs = preprocess(docs)

In [ ]:
rprint(preprocessed_docs[12])

Document(
    metadata={'source': 'https://n.news.naver.com/mnews/article/374/0000518676?sid=104'},
    page_content='닫기 [글로벌 비즈 브리핑] 오픈AI, 새 AI 모델 공개…美 정부 요청에 일부기관 제공 外 입력 
2026.06.29. 오전 3:29 수정 2026.06.29. 오전 5:48 기사원문 추천 반응 댓글 반응 닫기 글자 크기 변경하기 글자크기 
가1단계 작게 가2단계 보통 가3단계 크게 가4단계 아주크게 가5단계 최대크게 닫기 SNS 보내기 인쇄하기 [오픈AI 로고 
일러스트 (로이터=연합뉴스)][글로벌 비즈 브리핑] 한 눈에 보는 해외 경제 이슈▲메모리 대란에...애플 "中 칩 구해달라" 
美 정부에 로비▲오픈AI, 새 AI 모델 공개...美 정부 요청에 일부기관 제공▲"AI 붐, 장기 투자 실패로 끝날 위험:...섬뜩한 
경고▲한사람당 100억원 벌었다...日 키옥시아 직원들 \'잭팟\'▲스페이스X, 모바일 통신사업 진출 시동...차터와 
논의▲"비트코인, 소리없이 사라질 것"...닷컴버블 맞힌 그랜섬의 경고메모리 대란에...애플 "中 칩 구해달라" 美 정부에 
로비세계적인 메모리 반도체 품귀 탓에 생산 비용이 크게 늘어난 애플이 미국 정부의 블랙리스트에 오른 중국산 메모리 칩 
구매를 적극 검토하는 것으로 전해졌습니다.현지시간 26일 파이낸셜타임스(FT)는 애플이 미국 상무부를 비롯한 도널드 
트럼프 행정부 관계자들을 상대로 중국산 메모리 칩 구매를 승인해달라는 로비를 벌이고 있다고 보도했습니다.애플이 
관심을 보이는 중국의 반도체 업체는 창신메모리테크놀로지(CXMT)인 것으로 알려졌습니다.D램 제조사인 CXMT는 중국 
인민해방군과 연계됐다는 이유로 미 국방부가 ‘중국 군사기업(1260H)’이라는 일종의 블랙리스트에 올려놓은 업체입니다.이 
명단에 포함될 경우 직접적인 법적 제재를 받습니다. 하지만 거래 기업은 평판 리스크를 감수해야 합니다.애플이 워싱턴 
정가를 상대로 로비에 나선 것은 메모리 가격 급등으로 공급망 다변화를 추진해야 하는 상황이기 때문입니다.그러나 애플의
로비가 성공할지 여부는 미지수입니다. 특히 미국 의회의 반발이 변수로 꼽힙니다.하원 중국특별위원회의 존 
물레나(공화·미시간) 위원장은 “애플이 중국 군사 기업과 협력하는 것은 심각한 실수가 될 것”이라며 “중국 공산당이 핵심 
공급망을 장악하도록 돕는 결과를 초래할 수 있다”고 비판했습니다.마코 루비오 국무장관도 상원의원이던 지난 2022년 
애플이 중국의 YMTC 메모리 칩 채택을 검토했을 당시 “불장난을 하는 것”이라며 강하게 반대한 바 있습니다.낸드플래시 
메모리 전문 업체인 YMTC는 중국 군사기업 블랙리스트에 올라와 있는 업체입니다.오픈AI, 새 AI 모델 공개...美 정부 
요청에 일부기관 제공오픈AI가 차세대 인공지능(AI) 모델 \'GPT-5.6\' 제품을 출시하면서, 미국 정부의 요청으로 일부 
기관에만 제한적으로 우선 제공하기로 했습니다.오픈AI는 새 AI 모델 \'GPT-5.6\'을 \'솔\', \'테라\', \'루나\' 등 세 
가지 세부모델로 나눠 개발했다고 현지시간 26일 밝혔습니다.이 가운데 상위 모델인 \'솔\' 모델은 코딩, 생물학, 사이버 
보안 등 부문에서 역대 최고 성능과 자율적 에이전트 능력을 갖췄다고 오픈AI는 설명했습니다.오픈AI는 이 모델의 추론 
능력을 강화하기 위해 모델에 역대 가장 긴 시간을 부여하는 \'최대 추론 노력\' 옵션과 하위 에이전트를 활용해 복잡한 
작업을 좀 더 빠르게 수행하는 \'울트라 모드\'도 새로 도입했습니다.오픈AI는 터미널 코딩 능력을 재는 \'터미널-벤치 
2.1\' 성능지표(벤치마크)에서 GPT-5.6 솔 울트라가 91.9%, GPT-5.6 솔이 88.8%를 기록해 라이벌 앤트로픽의 클로드 
미토스5의 88%를 능가했다고도 강조했습니다.특히 오픈AI는 솔 모델이 현재까지 가장 견고한 안전 체계를 갖춰 고위험 
활동이나 해킹에 악용될 우려가 있는 민감한 사이버 요청, 오용 등에 대한 보호 조치를 강화했다고도 
강조했습니다.오픈AI는 그간 새 모델 발표 때마다 제공해 왔던 세부적인 성능지표는 공개하지 않았습니다.중위 세부 모델인
\'테라\'는 전작인 GPT-5.5와 유사한 성능을 보이면서도 가격은 절반으로 낮췄고, 하위 모델인 \'루나\'는 최저 비용으로 
준수한 성능을 제공한다고 밝혔습니다.오픈AI는 이들 모델을 우선 정부와 공유된 \'신뢰할 수 있는 파트너\'를 대상으로 선
제공한다면서 실제 일반 배포 시기는 몇 주 뒤가 될 것이라고 예고했습니다.다만 오픈AI는 "이러한 형태의 정부 승인 
절차가 장기적인 표준이 되어서는 안 된다고 생각한다"며 "해당 도구가 필요한 사용자·개발자·기업·사이버 보안 
전문가·글로벌 파트너들이 최고의 도구를 활용하지 못하게 막기 때문"이라고 덧붙였습니다.도널드 트럼프 대통령은 이달 초
AI 기업이 새 모델을 공개하기 최대 30일 전에 정부에 제출할 수 있도록 하는 내용의 행정명령에 서명했고, 미 상무부는 
앤트로픽의 첨단 모델 \'미토스5\'와 \'페이블5\'가 국가 안보 위험을 초래한다며 수출 통제 지침을 내놨습니다.한편 
오픈AI는 당초 올해로 예상됐던 상장 추진 일정을 늦추고 있는 것으로 전해졌습니다.이달 초 증권거래위원회(SEC)에 비공개
기업공개(IPO) 신청서를 제출한 오픈AI는 아직 잠재적인 공모 가격과 수요 등을 논의하기 위한 사전 회의도 열지 않았고, 
상장 일정에 대한 공식 계획도 제시하지 않았다고 미 경제방송 CNBC가 소식통을 인용해 이날 보도했습니다.앞서 
뉴욕타임스(NYT)는 오픈AI가 연내 추진하려 했던 상장 시기를 내년으로 미루는 쪽으로 기울고 있다고 전날 보도한 바 
있습니다."AI 붐, 장기 투자 실패로 끝날 위험:...섬뜩한 경고빅테크의 인공지능(AI) 투자 광풍이 장기적인 투자 붕괴로 
끝날 위험이 있다고 국제결제은행(BIS)이 경고했습니다.28일(현지시간) 파이낸셜타임스(FT)에 따르면 BIS는 이날 ‘연례 
경제 보고서’를 통해 기술 부문의 수익률이 기대에 못 미칠 가능성이 커질 경우 투자자들이 AI 기업에 대한 자금 공급을 
급격히 줄일 수 있다고 내다봤습니다. 5대 대형 하이퍼스케일러(초대형 클라우드 사업자)들은 2025년부터 2026년 말까지 
1조달러 이상을 투자할 것으로 예상됩니다.BIS는 현재의 AI 호황이 안고 있는 위험을 설명하며 “수익률에 대한 실망은 자금
조달의 갑작스러운 후퇴를 촉발할 수 있고, 자본지출 붐을 장기적인 투자 침체로 바꿀 수 있다”며 “이는 금융 여건에도 
연쇄적인 영향을 미칠 수 있다”고 분석했습니다.이는 AI 투자를 위한 주식·채권 발행 규모가 커지고, 이로 인해 글로벌 
시장의 변동성이 확대되는 데 대한 우려가 커지는 가운데 나왔습니다. 기술기업들은 AI 프로젝트 자금을 마련하기 위해 
신용시장으로 몰려들어 수천억달러를 조달했습니다.사상 최고 수준의 주가도 이들을 미국 주식시장으로 끌어들이고 
있습니다. 이달 초 스페이스X가 860억달러 규모의 초대형 기업공개(IPO)를 단행한 것은 AI 관련 주식에 대한 폭발적인 
수요를 보여주는 상징적 사례입니다.투자자들은 AI 투자가 충분한 수익을 내지 못할 경우 이 같은 채권 발행 러시가 역풍을
초래할 수 있다고 경고해왔습니다. 스페이스X IPO 이후 주식시장은 변동성이 커졌고, 연방준비제도(연준, Fed)의 금리 인상
전망에 무게가 실리고 있습니다. 일각에선 스페이스X가 IPO 직후 250억달러 규모의 채권 발행에 나선 것을 두고 “시장이 
버블 영역에 진입했다는 신호”라고 경고했습니다.BIS는 지금까지 AI에 대한 낙관론이 세계 성장에 중요한 순풍으로 
작용했다고 짚었습니다. 보고서는 AI가 기업들에 제공할 수 있는 효율성 향상을 감안하면, 향후 10년 동안 생산성을 상당히
끌어올릴 가능성이 있다고 인정했습니다.그럼에도 BIS는 과거 투자 붐 사례들이 “시사하는 바가 크다”고 지적했다. 예로는 
1830년대 운하 건설 확대, 1840년대 영국 철도 붐, 1990년대 말 닷컴 붐을 들었습니다.BIS는 “이들 사례는 

불러온 텍스트 데이터는 파일로 저장할 수 있습니다.

In [ ]:
rprint(preprocessed_docs[12].metadata)

{'source': 'https://n.news.naver.com/mnews/article/374/0000518676?sid=104'}

In [94]:
!pip install jsonlines langchain langchain-core

In [95]:
# 불러온 document 저장하기

import jsonlines

def save_docs_to_jsonl(documents, file_path):
    with jsonlines.open(file_path, mode="w") as writer:
        for doc in documents:
            writer.write(doc.model_dump())

save_docs_to_jsonl(preprocessed_docs, "docs.jsonl")

# jsonl 파일 불러오기
#from langchain.schema import Document
from langchain_core.documents import Document

def load_docs_from_jsonl(file_path):
    documents = []
    with jsonlines.open(file_path, mode="r") as reader:
        for doc in reader:
            documents.append(Document(**doc))
    return documents

평가 데이터와의 연결을 위해, 전처리한 데이터 대신 실습 시트의 데이터를 사용하겠습니다.

In [96]:
preprocessed_docs = load_docs_from_jsonl("docs.jsonl") #("eval.jsonl")

In [ ]:
preprocessed_docs[2]

Document(metadata={'source': 'https://n.news.naver.com/mnews/article/056/0012206804?sid=104'}, page_content='닫기 미국, AI 경쟁의 심장부…구글 데이터센터 첫 공개 입력 2026.06.26. 오후 12:44 수정 2026.06.26. 오후 12:52 기사원문 추천 반응 댓글 반응 닫기 글자 크기 변경하기 글자크기 가1단계 작게 가2단계 보통 가3단계 크게 가4단계 아주크게 가5단계 최대크게 닫기 SNS 보내기 인쇄하기 철저한 보안 절차를 통과해 들어선 미국 앨라배마주의 구글 AI 데이터센터. 끝이 보이지 않는 서버마다 수천 개의 첨단 반도체가 24시간 쉬지 않고 돌아갑니다.[카린 리들리/구글 사이트 운영 매니저 : "G메일을 확인하거나 유튜브 영상을 보고 지도를 검색하는 모든 작업이 바로 이 서버에서 처리됩니다."] AI 수요가 폭증하면서 빅테크 기업들의 데이터센터 확장 경쟁도 갈수록 치열해지고 있습니다. 구글 역시 앨라배마의 240만㎡ 규모 데이터센터 단지를 확장하고 있습니다. 하지만 막대한 전력과 물 사용, 소음 문제를 둘러싼 우려도 커지고 있습니다. 최근 미국 여론조사에서는 응답자의 70% 이상이 자신의 지역에 AI 데이터센터가 들어서는데 반대한다고 답했습니다.[토머스 갬블/구글 데이터센터 관리자 : "부정적인 시선이 많습니다. 그래서 전력, 물과 자원을 최대한 효율적으로 활용하는 방안을 찾고 있습니다."] 구글은 자체 변전소를 구축해 전력 비용 부담 논란을 줄이겠다고 밝혔지만, AI 데이터센터를 둘러싼 논란은 계속되고 있습니다. ■ 제보하기▷ 전화 : 02-781-1234, 4444▷ 이메일 : kbs1234@kbs.co.kr▷ 카카오톡 : \'KBS제보\' 검색, 채널 추가▷ 유튜브, 다음에서도 KBS뉴스를 구독해주세요! Copyright ⓒ KBS. All rights reserved. 무단 전재, 재배포 및 이용(AI 학습 포함) 금지 이 기사는 언론사에서 세계

## Chunking: 청크 단위로 나누기   



전처리가 완료된 docs를 chunk 단위로 분리합니다.
`chunk_size`와 `chunk_overlap`을 이용해 청크의 구성 방식을 조절할 수 있습니다.  

Chunk Size * K(검색할 청크의 수) 의 결과가 Context의 길이가 됩니다.

In [97]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
#from langchain import hub

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
# 0~1000, 800~1800, 1600~2600, ...
# 엔터, 공백, 쉼표 등의 구분자(delimiter)를 기준으로 최대한 청킹
chunks = text_splitter.split_documents(preprocessed_docs)
print(len(chunks))

# Top 5 검색이라면? --> 5 * 1000 = 5000 글자 Context

1030


## Vector Database 만들기   

구성된 청크를 벡터 데이터베이스에 로드합니다.   

In [98]:
from langchain_chroma import Chroma
import uuid

Chroma().delete_collection() # (메모리에 저장하는 경우) 기존 데이터 삭제

uuidstr = str(uuid.uuid4())[0:6]
# 랜덤 식별자

# DB 구성하기
db = Chroma(embedding_function=openai_embeddings,
            persist_directory=f"./chroma_OpenAI_{uuidstr}",
            # 파일 시스템에 저장 (생략시 메모리에 저장)

            collection_name='Web', # 식별 이름

            collection_metadata={'hnsw:space':'l2'},
            # l2 메트릭 설정(기본값, cosine, mmr 로 변경 가능)
            )

DB에 document를 추가합니다.    
OpenAI 임베딩은 30만 토큰 동시 처리 제한이 있어, 나눠서 전달합니다.

In [99]:
from tqdm import tqdm

print(len(chunks))
# 300,000 토큰 제한

# 100개씩 추가
for i in tqdm(range(0, len(chunks), 100)):
    db.add_documents(chunks[i:min(i+100, len(chunks))])

1030


100%|██████████| 11/11 [00:30<00:00,  2.77s/it]


db로부터 retriever를 구성합니다.

In [100]:
# Top 5 Search(기본값은 4)
retriever = db.as_retriever(search_kwargs={'k':5})

In [101]:
retriever.invoke("도메인 특화 언어 모델")

[Document(id='2d09eddc-92eb-47ec-8174-3028a44e38a8', metadata={'source': 'https://n.news.naver.com/mnews/article/014/0005475295?sid=105'}, page_content='닫기 이스트소프트, ‘제주어 특화 LLM’ 공개…국내 맞춤 AX 기반 구축 입력 2026.02.09. 오전 9:01 기사원문 추천 반응 댓글 반응 닫기 글자 크기 변경하기 글자크기 가1단계 작게 가2단계 보통 가3단계 크게 가4단계 아주크게 가5단계 최대크게 닫기 SNS 보내기 인쇄하기 [파이낸셜뉴스] 이스트소프트가 지역 특화 언어 모델인 ‘앨런 LLM 제주어 v1 4B(Alan LLM Jeju Dialect v1 4B)’를 세계 최대의 오픈소스 플랫폼 허깅페이스에 공개했다고 9일 밝혔다. 이번 제주어 특화 LLM 공개는 국내 환경에 최적화된 기술 고도화를 통해 대국민 AX 기반을 다지기 위해 진행됐다. 특히 한국적 특성이 반영된 데이터를 정교하게 학습시켜 실제 현장에서 즉시 활용 가능한 ‘국내 맞춤형 AX’ 기술력을 확보했다는 점에서 의미가 크다. 이 모델은 오픈소스 기반의 고도화된 미세 조정(Fine-tuning) 기법을 적용해, 40억 개의 파라미터라는 효율적인 규모 내에서 제주어와 표준어 사이의 정교한 소통을 구현했다. 무엇보다 4B 규모의 경량 모델이라 하더라도 타겟 언어와 특정 도메인에 특화될 경우, 수백억 파라미터의 거대 모델 성능을 능가할 수 있음을 이번 공개를 통해 입증했다. 이스트소프트는 이러한 강력한 파인튜닝 역량을 바탕으로 향후 독자 AI 파운데이션 모델과 시너지를 창출하고, 다양한 지역 인프라와 결합해 관광·행정 등 다방면에서 실질적인 지역 특화 AX의 성공 사례를 만들어갈 방침이다. 특히 제주국제자유도시개발센터(JDC)와 ‘제주 지역의 AI 전환 및 글로벌 경쟁력 강화를 위한 업무협약’을 체결하는 등 자사의 제주 캠퍼스를 중심으로 한 활동도 활발히 이루어지고 있어, 

## Prompting

RAG를 위한 간단한 프롬프트를 작성합니다.

In [102]:
from langchain_core.prompts import ChatPromptTemplate

In [103]:
prompt = ChatPromptTemplate([
    ("system", '''당신은 QA(Question-Answering)을 수행하는 Assistant입니다.
다음의 Context를 이용하여 Question에 답변하세요.
정확한 답변을 제공하세요.
만약 모든 Context를 다 확인해도 정보가 없다면,
"정보가 부족하여 답변할 수 없습니다."를 출력하세요.
Question과 무관한 내용이 들어 있을 수 있으므로 신중히 판단하세요.'''),
('human','''
---
Context: {context}
---
Question: {question}''')])
prompt.pretty_print()

================================ System Message ================================

당신은 QA(Question-Answering)을 수행하는 Assistant입니다.
다음의 Context를 이용하여 Question에 답변하세요.
정확한 답변을 제공하세요.
만약 모든 Context를 다 확인해도 정보가 없다면,
"정보가 부족하여 답변할 수 없습니다."를 출력하세요.
Question과 무관한 내용이 들어 있을 수 있으므로 신중히 판단하세요.

================================ Human Message =================================


---
Context: {context}
---
Question: {question}


## Chain

RAG를 수행하기 위한 Chain을 만듭니다.

RAG Chain은 프롬프트에 context와 question을 전달해야 합니다.    
체인의 입력은 Question만 들어가므로, Context를 동시에 prompt에 넣기 위해서는 아래의 구성이 필요합니다.

In [104]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


# retriever의 결과물은 List[Document] 이므로 이를 ---로 구분하는 함수
# metadata의 source를 보존하여 추가
def format_docs(docs):
    return " \n\n---\n\n ".join(['URL: '+ doc.metadata['source'] + '\nContent:' + doc.page_content for doc in docs])
    # join : 구분자를 기준으로 스트링 리스트를 하나의 스트링으로 연결


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    # retriever : question을 받아서 context 검색: document 반환
    # format_docs : document 형태를 받아서 텍스트로 변환
    # RunnablePassthrough(): 체인의 입력을 그대로 저장
    | prompt
    | llm
    | StrOutputParser()
)

In [105]:
print(format_docs(retriever.invoke("알리바바의 언어 모델 이름은?")))

URL: https://n.news.naver.com/mnews/article/020/0003684853?sid=105
Content:닫기 [기술영업人] 플리토 김재훈 팀장이 개척한 ‘언어 데이터 세일즈’의 길 입력 2025.12.26. 오전 11:55 기사원문 추천 반응 댓글 반응 닫기 글자 크기 변경하기 글자크기 가1단계 작게 가2단계 보통 가3단계 크게 가4단계 아주크게 가5단계 최대크게 닫기 SNS 보내기 인쇄하기 ※ 21세기는 기술경쟁의 시대입니다. 수많은 빅테크 기업이 지금 이 순간도 기술과 제품을 놓고 전 세계 산업 현장에서 경쟁합니다. 그리고 이 경쟁의 현장이 바로 기술영업입니다. 기술영업은 기술적 이해가 필요한 영업으로, 주로 기업 대 기업 간 영업에서 이뤄집니다. 기술영업 전선에서는 기업의 기술력과 실력으로 경쟁하고, 그 결과가 IT 기업이 움직이는 원동력이 됩니다. 오늘날 현장에서 기술영업人들이 어떻게 경쟁하는지, 기술과 기업, 사람을 들여다보겠습니다.오늘날 AI라고 하면 챗GPT 같은 대형언어모델(LLM)을 떠올리는 사람이 많고, 실제로도 많은 AI가 LLM 모델을 기반으로 구현된다. LLM의 발전이 곧 AI의 발전이다보니 자연스레 ‘언어데이터’가 핵심 자산으로 떠올랐다. LLM 데이터의 약 절반은 언어 기반의 일반 지식 데이터고, 25%는 수학 및 추론, 25%는 코드 및 다국어 데이터 등으로 구성된다. 초창기에는 가능한 많은 매개변수와 데이터를 확보해 성능을 끌어올리는 방식이 대세였지만, 잦은 환각 현상과 데이터 비대화를 막기 위해 지금은 고품질의 정련된 언어 데이터를 활용하는 게 기본이다.이에 따라 AI 학습용 언어 데이터를 취급하는 기업, 특히 기존 번역 업계에 있던 기업들이 AI의 핵심으로 자리 잡았다. 미국의 트랜스퍼펙트는 2020년 ‘데이터포스’라는 데이터 브랜드를 출범하며 AI 언어 데이터 시장의 강자로 등극했고, 영국의 RWS나 캐나다의 텔러스 디지털도 번역에서 데이터 기업으로 변모했다. 스타트업으로는 스케일AI나 디파인드

In [ ]:
rag_chain.invoke("도메인 특화 언어 모델이란 무엇입니까? 어떤 예시가 있나요?")

'**도메인 특화 언어 모델이란?**\n\n- **정의**  \n  특정 산업·분야(예: 의료, 법률, 특허, 지역 방언 등)의 전문 용어나 문체, 지식 구조를 반영하도록 사전 학습 또는 미세조정(fine‑tuning)된 대형 언어 모델(LLM)을 말합니다.  \n- **특징**  \n  - *도메인 전용 데이터* (전문 기사, 법령·특허 문서, 지역 방언 텍스트 등)를 사용해 학습 → 해당 분야에서의 정확도와 효율이 일반 LLM보다 높음  \n  - *작은 규모(4B~40억 파라미터) 모델**이라도 도메인 특화 덕분에 수백억 파라미터 대형 모델을 능가할 수 있음  \n  - 기업 내부의 AI 전환(AI\u202fTransformation, AX) 전략에서 “기술 도입”이 아니라 *가치 사슬 재편*을 위한 핵심 역량으로 활용  \n\n---\n\n**대표적인 예시**\n\n| # | 사례 | 주요 내용 |\n|---|------|-----------|\n| 1 | **Eastsoft – Alan LLM Jeju Dialect v1\u202f4B** | 제주어와 표준어 사이의 정교한 소통을 위해 40억 파라미터 규모로 미세조정. 지역 특화 AI를 통해 관광·행정 등 현장 서비스에 바로 적용 가능. |\n| 2 | **Trillium Labs – 도메인 특화 증류 기반 소형 LLM** | 의료, 법률, 과학, 행정 등 특정 영역별 ‘전문가 모델’을 개별 증류 후 MoE(Expert‑Mixing) 구조로 결합해 추론 효율과 성능을 동시에 향상. 국가 전략기술로 인정받음. |\n| 3 | **한국 특허 데이터 기반 도메인 모델** | 한국 특허청에 보유된 106개국·1억\u202f7천만 건의 출원·등록 데이터(2,500억 개 문장, 16억 장 도면)를 학습 자산으로 활용해 권리·계약·청구항 분석까지 수행. |\n| 4 | **기타** | - AI 파운데이션이 산업에 적용될 때 ‘권리 처리’와 같은 특허·IP 데이터를 포함한 도메인 모델이 필요함.<br>- 국내 기업들

In [ ]:
rag_chain.invoke("인공지능의 최근 발전 방식은? 관련 링크도 보여주세요")

In [ ]:
rag_chain.invoke("알리바바가 개발한 언어 모델 이름은?")

만약 Context가 포함된 RAG 결과를 보고 싶다면, RunnableParallel을 사용하면 됩니다.

assign()을 이용하면, 체인의 결과를 받아 새로운 체인에 전달하고, 그 결과를 가져옵니다.

In [106]:
# assign : 결과를 받아서 새로운 인수 추가하고 원래 결과와 함께 전달

from langchain_core.runnables import RunnableParallel

rag_chain_from_docs = (
    prompt
    | llm
    | StrOutputParser()
)

rag_chain_with_source = RunnableParallel(
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
).assign(answer=rag_chain_from_docs)

rag_chain_with_source.invoke("인공지능의 최근 발전 방식은? 관련 링크도 보여주세요")

# retriever가 1번 실행됨
# retriever의 실행 결과를 rag_chain_from_docs 에 넘겨주기 때문에

{'context': 'URL: https://n.news.naver.com/mnews/article/277/0005782636?sid=101\nContent:닫기 챗GPT 독주 끝났나…3강 구도로 재편 입력 2026.06.29. 오전 10:16 수정 2026.06.29. 오전 10:17 기사원문 추천 반응 댓글 반응 닫기 글자 크기 변경하기 글자크기 가1단계 작게 가2단계 보통 가3단계 크게 가4단계 아주크게 가5단계 최대크게 닫기 SNS 보내기 인쇄하기 마인드로직, 생성형 AI 플랫폼 ‘팩트챗’ 데이터 공개앤트로픽과 구글이 우위였던 오픈AI를 상대로 점유율을 빠르게 끌어올리며 생성형 멀티 인공지능(AI) 시장이 3강 구도로 재편되고 있다는 데이터가 공개됐다.마인드로직은 자사 멀티 인공지능(AI) 에이전트 플랫폼 \'팩트챗(FactChat)\' 이용 데이터를 기반으로 한 국내 생성형 AI 활용 패턴 분석 결과를 29일 공개했다. 팩트챗은 GPT, 클로드, 제미나이 등 약 80여 개의 생성형 AI 모델을 통합 제공하는 플랫폼이다. 현재 국내 주요 대학과 초·중·고교 교육기관 및 공공기관 등에 잇따라 도입되고 있으며 약 50만 명의 유료 이용자를 확보했다.이번에 공개된 데이터는 마인드로직이 지난해 7월부터 올해 6월까지 국내 팩트챗 이용자의 AI 활용 패턴을 분석한 결과다. 조사 결과 앤트로픽의 \'클로드\'의 약진이 눈에 띄었다. 클로드는 지난 5월 처음으로 GPT의 사용 비중을 앞질렀다. 오픈AI의 GPT 모델 이용 비중은 지난해 9월 85.7%에서 올해 5월 34.8%까지 감소한 반면, 클로드는 같은 기간 5.7%에서 36%까지 급등했다. 클로드 시리즈의 코딩 역량 등이 높은 평가를 받으며 국내 이용자들을 대거 흡수한 것으로 풀이된다. 구글의 제미나이 역시 지난해 9월 6.9%에서 올해 1월 31%로 사용 비중이 확대된 뒤, 지난 5월까지 20%대의 점유율을 유지했다. 마인드로직은 국내 이용자들의 AI 활용 능력이 향상된 결과라고 설명했다. 생성형 AI 이

In [107]:
# Runnable Quiz

runnable = RunnableParallel(
    passed=RunnablePassthrough(),
    extra=RunnablePassthrough.assign(mult=lambda x: x["num"] * 3),
    modified=lambda x: x["num"] + 1,
)

runnable.invoke({"num": 1})
# 결과 이해해보기!

{'passed': {'num': 1}, 'extra': {'num': 1, 'mult': 3}, 'modified': 2}

이번에는 오픈 모델을 사용합니다.   
오픈 임베딩을 통해 구성한 DB와 원래 DB를 비교해 보겠습니다.

In [ ]:
uuidstr = str(uuid.uuid4())[0:6]

open_db = Chroma(embedding_function=open_embeddings, # qwen 3 emb 0.6b
                           persist_directory=f"./chroma_open_{uuidstr}", # 별도 폴더에 저장
                           collection_name='Web', # 식별 이름
                           collection_metadata={'hnsw:space':'l2'}
                           )

# 10개씩 추가
for i in tqdm(range(0, len(chunks), 10)):
    open_db.add_documents(chunks[i:min(i+10, len(chunks))])

100%|██████████| 100/100 [00:54<00:00,  1.84it/s]


In [ ]:
open_retriever = open_db.as_retriever(search_kwargs={'k':5})

In [ ]:
open_retriever.invoke("알리바바의 언어 모델은 무엇입니까?")

In [108]:
rag_chain_open = (
    {"context": open_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [109]:
rag_chain_from_docs = (
    prompt
    | llm
    | StrOutputParser()
)

rag_chain_with_source_open = RunnableParallel(
    {"context": open_retriever | format_docs, "question": RunnablePassthrough()}
).assign(answer=rag_chain_from_docs)

In [110]:
questions = ["도메인 특화 언어 모델이란 무엇입니까? 어떤 예시가 있나요?",
             "인공지능의 최근 발전 방식은? 관련 링크도 보여주세요",
             "알리바바의 언어 모델 이름은?"]

In [111]:
# Retriever 비교

result_open = open_retriever.batch(questions)

for i in range(3):
    print(f'Question: {questions[i]}')
    for j in result_open[i]:
        print('   ',j.page_content[0:50])

Question: 도메인 특화 언어 모델이란 무엇입니까? 어떤 예시가 있나요?
    닫기 LGU+, 세계 최고 권위 AI 학회에 익시젠 논문 채택…“산업 특화 기술력 입증” 
    닫기 이스트소프트, ‘제주어 특화 LLM’ 공개…국내 맞춤 AX 기반 구축 입력 2026.
    닫기 이스트소프트, 제주 특화 AI 언어모델 오픈소스 공개 입력 2026.02.09. 오후
    닫기 LGU+ '익시젠' 논문, 글로벌 학회 채택…산업 특화 기술력 입증 입력 2025.1
    전면에 나섰다"며 "단일 모델 의존 자체가 핵심 리스크"라고 진단했다. 이어 "업무 유형과
Question: 인공지능의 최근 발전 방식은? 관련 링크도 보여주세요
    AI 기업들이 개발 과정의 가치 체계를 더 명확히 공개해야 한다는 목소리가 커지고 있는 상
    ⓒ 전자신문. All rights reserved. 무단 전재 및 재배포 금지. 이 기사는
    메인에서 만나보세요! 구독중 언론사의 주요 뉴스를 메인에서 바로 만나보세요!주요 뉴스를 메
    “최근 궁금한 것이 생기면 포털 검색 대신 AI에 물어볼 정도로 AI의 영향력이 커졌기 때
    경제 섹션으로 분류했습니다. 기사 섹션 분류 안내 기사의 섹션 정보는 해당 언론사의 분류를
Question: 알리바바의 언어 모델 이름은?
    닫기 이스트소프트, 제주 특화 AI 언어모델 오픈소스 공개 입력 2026.02.09. 오후
    닫기 이스트소프트, ‘제주어 특화 LLM’ 공개…국내 맞춤 AX 기반 구축 입력 2026.
    닫기 언어모델 솔루션 고도화… 기업 기술자산 유출 위험 차단 입력 2026.05.27. 오
    언어 모델)을 만들려 한다. 이런 분야에도 협력해 지원 분야를 넓히고 강점을 가져가고 싶다
    만들었다"며 "블라인드 평가에서 앤트로픽 '소넷 4.6'보다 높은 선호도 점수를 받았다"고


In [112]:
result_openai = retriever.batch(questions)
result_openai
for i in range(3):
    print(f'Question: {questions[i]}')
    for j in result_openai[i]:
        print('   ',j.page_content[0:50])

Question: 도메인 특화 언어 모델이란 무엇입니까? 어떤 예시가 있나요?
    닫기 이스트소프트, ‘제주어 특화 LLM’ 공개…국내 맞춤 AX 기반 구축 입력 2026.
    닫기 AI 모델 최적화도 국가전략기술 인정 입력 2025.12.18. 오후 12:01 기사
    닫기 이스트소프트, 제주 특화 AI 언어모델 오픈소스 공개 입력 2026.02.09. 오후
    것은 단순한 텍스트가 아니라 AI 파운데이션이 산업으로 내려갈 때 부딪히는 권리 처리의 자
    전면에 나섰다"며 "단일 모델 의존 자체가 핵심 리스크"라고 진단했다. 이어 "업무 유형과
Question: 인공지능의 최근 발전 방식은? 관련 링크도 보여주세요
    닫기 챗GPT 독주 끝났나…3강 구도로 재편 입력 2026.06.29. 오전 10:16 수
    앤트로픽[로이터=연합뉴스. 재판매 및 DB 금지] 이들이 이직한 앤트로픽과 오픈AI는 현재
    닫기 단일 AI 의존 탈피…알아서 조합하는 '에이전트형 AI' 뜬다 입력 2026.06.2
    6.9%에서 올해 1월 31%로 사용 비중이 확대된 뒤, 지난 5월까지 20%대의 점유율을
    다른 연구소 모델을 증류하지 않고 라이선스가 없거나 불투명한 데이터에 의존하지 않는다"며 
Question: 알리바바의 언어 모델 이름은?
    닫기 [기술영업人] 플리토 김재훈 팀장이 개척한 ‘언어 데이터 세일즈’의 길 입력 2025
    닫기 “검색이 아니라 AI가 산다”…쇼핑 판 뒤집는 AI 에이전트 입력 2026.06.28
    닫기 이스트소프트, ‘제주어 특화 LLM’ 공개…국내 맞춤 AX 기반 구축 입력 2026.
    40% 이상으로 일반 이용자의 약 20%보다 두 배 가까이 높았다. AI가 제품 비교와 맞
    닫기 이스트소프트, 제주 특화 AI 언어모델 오픈소스 공개 입력 2026.02.09. 오후


In [113]:
# 최종 결과 비교
print(rag_chain_open.batch(questions))

['**도메인 특화 언어 모델(Domain‑Specific Language Model)**  \n- **정의**: 특정 산업·분야에서 사용되는 전문 용어나 문맥을 대규모 데이터로 추가 학습시켜, 그 분야에 맞는 높은 성능을 발휘하도록 만든 소형 또는 거대 언어 모델입니다.  \n- **특징**  \n  - 일반적인 범용 언어 모델(LLM)보다 해당 도메인에서 더 깊이 있는 지식을 주입할 수 있습니다.  \n  - ‘도메인 특화 학습(DACP)’ 같은 기법을 활용해 산업 데이터를 지속적으로 학습하면서도 기존의 일반 언어 능력은 보존합니다.  \n  - 파괴적 망각(Catastrophic Forgetting)을 방지하는 방법이 병행됩니다.\n\n**주요 예시**\n\n| 기업/프로젝트 | 모델명 및 규모 | 적용 도메인 | 성능 향상 |\n|----------------|---------------|-------------|-----------|\n| **LG유플러스 (LGU+)** | ixi‑GEN (sLLM) | 통신·금융 분야 | 통신: 약 56\u202f% / 금융: 약 39\u202f% 성능 향상 |\n| **이스트소프트** | Alan LLM 제주어 v1 4B (40억 파라미터) | 제주 지역 언어(제주어와 표준어 간 소통) | 경량 모델임에도 특정 도메인에서 대형 모델을 능가할 수 있음 |\n\n- **기타 사례**  \n  - Deloitte 컨설팅 기사에서는 광산 탐사, 제약 신약 개발, 금융 재무 운영 등 전문 비즈니스 과제에 특화된 소형 언어 모델(SLM)의 활용이 증가하고 있다고 언급했습니다.  \n\n즉, 도메인 특화 언어 모델은 특정 산업·분야의 요구를 충족시키기 위해 맞춤형 데이터와 학습 기법을 적용해 만든 모델이며, LGU+의 ixi‑GEN과 이스트소프트의 제주어 LLM이 대표적인 예시입니다.', '**인공지능의 최근 발전 방식**\n\n| # | 핵심 발전 방향 | 구체적 내용 | 관련 기사 링크 |\n|---|------------

In [115]:
rprint(rag_chain.batch(questions))

[
    '**도메인 특화 언어 모델이란?**\n\n도메인 특화 언어 모델(Domain‑Specific LLM)은 특정 분야·주제·지역에 대한 
데이터만을 사용해 학습하거나 미세조정(fine‑tuning)한 대규모 언어 모델입니다.  \n- **특징**  \n  - **전문성**: 
일반적인 범용 모델보다 해당 도메인(예: 의료, 법률, 과학, 행정, 지역 방언 등)에 특화된 지식과 어휘를 보유합니다.  \n
- **효율성**: 작은 규모의 파라미터(4B 이하)로도 큰 모델(수십~백억 파라미터)을 능가할 수 있는 성능을 발휘하도록 
설계됩니다.  \n  - **기술적 접근법**  \n    - *Fine‑tuning*: 기존 대형 모델에 도메인 데이터만 추가 학습시켜 
특화합니다.  \n    - *Distillation + Mixture‑of‑Experts (MoE)*: 소형 LLM을 여러 전문가(Expert) 모델로 분할해 각 
영역별 지식을 집중시키고, MoE 구조를 통해 추론 시 필요한 Expert만 활성화하여 효율성을 높입니다.\n\n**주요 
예시**\n\n| # | 도메인 | 모델명 / 기술 | 주요 내용 |\n|---|--------|--------------|-----------|\n| 1 | 제주 지역 
방언 | **Alan LLM Jeju Dialect v1\u202f4B** (이스트소프트) | 40억 개 파라미터 규모의 경량 모델로, 제주어와 표준어 
사이를 정교하게 소통하도록 학습. 오픈소스 플랫폼 허깅페이스에 공개. |\n| 2 | 의료·법률·과학·행정 등 전문 영역 | 
**도메인 특화 증류 기반 소형 LLM** (트릴리온랩스) | 각 영역별 데이터를 개별적으로 증류해 전문가 모델을 만들고, MoE 
구조로 결합해 추론 효율과 성능을 동시에 개선. 국가전략기술로 인정받음. |\n| 3 | 특허·산업 IP 데이터 | **특허 전문 
LLM** (민간 기업) | 106개국 1억\u202f7천만 건의 특허, 2,500억 개 문장, 16억 장 도면을 학습 자산으로 활용해 권리 
문서 정합성 검사 등까지 수행. |\n| 4 | 지역·행정·관광 서비스 | **제주 지역 AX(Artificial Intelligence 전환) 모델** 
(이스트소프트 + JDC 협업) | 제주국제자유도시개발센터와 업무협약을 통해 관광·행정 등 실질적인 지역 특화 AI 서비스를 
목표. |\n\n> **핵심 포인트**  \n> - 도메인 특화 언어 모델은 “특정 분야에 맞춘 데이터만으로 학습”함으로써, 범용 
모델보다 해당 영역에서 더 높은 정확도와 효율성을 제공합니다.  \n> - 작은 규모(4B 이하)라도 전문 지식이 집중되면 
대형 모델을 능가할 수 있다는 점이 최근 연구·산업계의 큰 관심사입니다.\n\n필요한 추가 정보나 다른 도메인에 대한 
예시를 원하시면 알려 주세요!',
    '**인공지능의 최근 발전 방식**\n\n| 구분 | 핵심 내용 | 대표 사례 | 관련 기사 링크 
|\n|------|-----------|-----------|----------------|\n| **멀티‑AI(다중 모델) 시대** | 한 가지 LLM에 의존하지 않고, 
사용 목적·작업 종류에 따라 여러 모델을 선택해 활용한다. | Anthropic의 Claude, Google Gemini, OpenAI GPT 등 80여 개 
모델이 통합된 “FactChat” 플랫폼 | <https://n.news.naver.com/mnews/article/277/0005782636?sid=101> |\n| **에이전트형
AI** | 사용자가 직접 모델을 지정하지 않아도, 질의 성격이나 목적에 따라 최적의 모델을 자동으로 조합해 실행한다. 
“슈퍼 에이전트(Super Agent)”가 대표적이다. | FactChat 내 슈퍼 에이전트(2026년 5월~6월 사용 비중 2.3%→11.6%) | 
<https://n.news.naver.com/mnews/article/018/0006317621?sid=105> |\n| **코딩·문서 분석 역량 강화** | Claude 시리즈가
코딩, 문서 분석에서 뛰어난 성능을 보이며 사용 비중이 급증했다. | 2026년 5월 Claude 점유율 36% (GPT 34.8%) | 
<https://n.news.naver.com/mnews/article/018/0006317621?sid=105> |\n| **시장 구조의 삼파전** | OpenAI(ChatGPT) 
독주가 끝나고 Anthropic(Claude), Google(Gemini)이 경쟁하며 “3강 구도”가 형성된다. | GPT 점유율 85%→34%, Claude 
5.7%→36%, Gemini 6.9%→31% | <https://n.news.naver.com/mnews/article/277/0005782636?sid=101> |\n| **데이터 기반 활용
패턴 분석** | 사용자가 실제로 어떤 모델을 언제 쓰는지 토큰 사용량 데이터를 통해 분석, AI 활용 능력 향상을 확인한다.
| FactChat 이용자 50만 명 데이터 분석 결과 발표 | <https://n.news.naver.com/mnews/article/018/0006317621?sid=105> 
|\n\n### 핵심 포인트\n- **단일 모델 독주 → 다중 모델 선택**  \n  GPT가 한때 압도적이었으나, 사용자가 목적에 따라 
Claude·Gemini 등 다양한 LLM을 자유롭게 고를 수 있게 됨.\n- **자동 최적화 기능**  \n  “슈퍼 에이전트”는 질의 유형을 
파악해 가장 적합한 모델을 자동으로 조합, 사용자는 모델 선택에 대한 부담이 줄어듦.\n- **코딩·문서 분석 전문성 부각**
\n  Claude 시리즈가 코딩과 문서 이해에서 강세를 보이며 시장 점유율 급증.\n- **삼파전 구조**  \n  OpenAI, Anthropic,
Google 세 기업이 경쟁하며 AI 생태계가 다변화되고 있음.\n\n> **관련 기사 링크**  \n> - 
<https://n.news.naver.com/mnews/article/277/0005782636?sid=101>  \n> - 
<https://n.news.naver.com/mnews/article/018/0006317621?sid=105>\n\n이와 같이 인공지능은 “멀티‑AI”와 “에이전트형 
AI”를 중심으로 발전하고 있으며, 사용자가 목적에 맞는 최적의 모델을 자동으로 선택해 활용하는 방향으로 진화하고 
있습니다.',
    '정보가 부족하여 답변할 수 없습니다.'
]

# RAG 성능 평가하기   

구성한 RAG의 성능은 어떻게 평가할까요?   

RAGAS (https://docs.ragas.io/en/stable/)는 다양한 메트릭을 통한 RAG의 성능 평가를 지원합니다.

RAG의 평가를 위해서는 정답이 있는 Q/A 데이터가 필요합니다.   
실습 시트에서 eval.jsonl을 다운로드하여 불러옵니다.

In [ ]:
import pandas as pd
df = pd.read_csv('./eval.csv')
eval_dataset = df.to_dict('list')
eval_dataset

In [ ]:
questions, ground_truths = eval_dataset['questions'], eval_dataset['ground_truths']

In [ ]:
for i in range(len(questions)):
    print(f'#{i}')
    print(f'Question: {questions[i]}\n')
    print(f'Ground Truth: {ground_truths[i]}\n')
    print('-----------')

구성된 RAG 체인을 이용해, RAGAS의 Evaluate에 필요한 데이터를 구성합니다.

In [ ]:
dataset = []

for query,reference in zip(questions,ground_truths):

    relevant_docs = [doc.page_content for doc in retriever.invoke(query)]
    response = rag_chain.invoke(query)
    dataset.append(
        {
            "user_input":query,
            "retrieved_contexts":relevant_docs,
            "response":response,
            "reference":reference
        }
        # 질문, 검색결과, 답변, 정답(레퍼런스)
    )

In [ ]:
from ragas import EvaluationDataset
evaluation_dataset = EvaluationDataset.from_list(dataset)
evaluation_dataset

RAGAS는 LLM을 이용해 정답과 답변을 개별 Claim(주장)으로 분할합니다.

이후, `LLMContextRecall`, `Faitufulness`, `FactualCorrectness` 등의 다양한 메트릭을 통해 RAG 파이프라인의 성능을 평가합니다.   
LLM 기반의 방법이므로 평가 LLM의 선정이 중요하며, 절대 수치보다는 상대적 비교가 효과적입니다.


- Context Recall: 정답의 Claim들이 모두 검색됐는가
- Faithfulness : 답변의 Claim이 얼마나 검색 결과에 근거했는가
- Factual Correctness : 정답과 답변의 Claim이 얼마나 일치하는가
- Bleu Score : 정답과 답변 키워드가 얼마나 일치하는가
- Semantic Sim : 정답 답변 임베딩이 얼마나 가까운가   


In [ ]:
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness
from ragas.metrics import BleuScore, SemanticSimilarity
from langchain_openai import ChatOpenAI

# https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/

# 평가자 LLM
evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))

# 평가자 Embedding
evaluator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model = 'text-embedding-3-large'))
semantic_scorer = SemanticSimilarity(embeddings = evaluator_embeddings)


result = evaluate(dataset=evaluation_dataset,
                  metrics=[BleuScore(), LLMContextRecall(), semantic_scorer, Faithfulness(), FactualCorrectness()],
                  llm=evaluator_llm,
                  embeddings = evaluator_embeddings
                  )
result

In [ ]:
result.scores

In [ ]:
result.to_pandas()

In [ ]:
# dict로 변경하기(복잡..)
import ast

result_str = str(result)
result_dict = ast.literal_eval(result_str)
result_dict

# [실습] 오픈 임베딩 체인의 성능 평가하기   

다음의 코드를 이용해, 오픈 임베딩을 비롯한 다양한 파이프라인의 모델 성능도 평가해 보세요.   

새로운 Ollama 모델을 쓰고 싶은 경우, `ollama pull`을 통해 저장하고   
`ollama stop`을 통해 기존 모델을 해제한 뒤 실행해 주세요.

In [ ]:
MODEL_NAME='gemma3:12b'
# 'gpt-'를 통해 gpt-oss 이외의 OpenAI 모델 불러올 수 있음

TEMPERATURE=0
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
TOP_K = 10
embedding_method = 'QWEN' # or 'QWEN'


# CSV 파일 경로 설정
csv_path = "experiments/experiment_log.csv"
os.makedirs("experiments", exist_ok=True)


def RAG_pipeline():

    emb_map = {'QWEN':open_embeddings, 'OPENAI':openai_embeddings}

    dataset = []

    if 'gpt' in MODEL_NAME:
        llm = ChatOpenAI(model=MODEL_NAME, temperature = TEMPERATURE)
    else:
        try: # Reasoning Model에 한해 Path 분리
            llm = ChatOllama(model=MODEL_NAME, temperature = TEMPERATURE, reasoning = True)
            llm.invoke("hello!")
        except:
            llm = ChatOllama(model=MODEL_NAME, temperature = TEMPERATURE)

    print('## LLM 세팅 완료')
    Chroma().delete_collection() # 메모리 기존 데이터 삭제

    preprocessed_docs = load_docs_from_jsonl("eval.jsonl")

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    print('## DB 구성 중...')

    chunks = text_splitter.split_documents(preprocessed_docs)
    print(f'## Chunks 생성 완료... 총 {len(chunks)} 개 청크')
    db = Chroma(embedding_function=emb_map[embedding_method],
                collection_metadata={'hnsw:space':'l2'},
                )

    for i in tqdm(range(0, len(chunks), 10)):
        db.add_documents(chunks[i:min(i+10, len(chunks))])

    print('## DB 구성 완료...')

    retriever = db.as_retriever(search_kwargs={'k':TOP_K})

    rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser())

    for query,reference in tqdm(zip(questions,ground_truths)):

        relevant_docs = [doc.page_content for doc in retriever.invoke(query)]
        response = rag_chain.invoke(query)
        dataset.append(
            {
                "user_input":query,
                "retrieved_contexts":relevant_docs,
                "response":response,
                "reference":reference
            }
        )

    print('## 답변 생성 완료...')
    evaluation_dataset = EvaluationDataset.from_list(dataset)

    result = evaluate(dataset=evaluation_dataset,
                    metrics=[BleuScore(), LLMContextRecall(), semantic_scorer, Faithfulness(), FactualCorrectness()],
                    llm=evaluator_llm,
                    embeddings = evaluator_embeddings
                    )

    result_str = str(result)
    result_dict = ast.literal_eval(result_str)
    return result_dict


실험 결과를 저장할 수 있습니다.

In [ ]:
import csv
import datetime
import os

result = RAG_pipeline()

timestamp = datetime.datetime.now().isoformat()
row = {
    "timestamp": timestamp,
    "model_name": MODEL_NAME,
    "embedding_method": embedding_method,
    "temperature": TEMPERATURE,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "top_k": TOP_K,
    **result
}

# 파일이 없으면 헤더 추가
write_header = not os.path.exists(csv_path)

with open(csv_path, "a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=row.keys())
    if write_header:
        writer.writeheader()
    writer.writerow(row)

print(f"[{timestamp}] {MODEL_NAME} {embedding_method}| temp={TEMPERATURE}, chunk={CHUNK_SIZE}/{CHUNK_OVERLAP}, top_k={TOP_K} | results={result}")
